In [1]:
# %load_ext autoreload
# %autoreload 2

import numpy as np
from scipy import spatial as sp
from itertools import product, permutations
from matplotlib import pyplot as plt

from transformation import Transformation
from point_cloud import PointCloud
from optimizers import diam
from eucl_haus import approx_eucl_haus


Let $A, B \in \mathbb{R}^k$ be the two point clouds for $k \in \{2, 3\}$ centered at the origin, and let $d_\max = \max\{\text{diam} A, \text{diam} B\}$. Because the distance from the origin to the most extreme point of $A\cup B$ is $< d_\max$, which is invariant to rotation or reflection, we never need to translate $A$ by more than $d_\max$. This implies that the search space $\Omega$ for $k=2$ is (a subset of) $[-d_\max, d_\max]^2 \times [0, 2\pi) \times \{0, 1\}$.

We speed up computing $d_\text{H}\big(T(A), B\big)$ for each $T \in \Omega$ to be $O(n \log n)$ on average by noting that $$d_\text{H}\big(T(A), B\big) = \max\Big\{\overrightarrow{d_\text{H}}\big(T(A), B\big), \overrightarrow{d_\text{H}}\big(B, T(A)\big)\Big\} = \max\Big\{\overrightarrow{d_\text{H}}\big(T(A), B\big), \overrightarrow{d_\text{H}}\big(T^{-1}(B), A\big)\Big\}$$ and setting up a k-d tree once for each space.

In [2]:
def diam(coords):
    hull = sp.ConvexHull(coords)
    hull_coords = coords[hull.vertices]
    candidate_distances = sp.distance.cdist(hull_coords, hull_coords)
    
    return candidate_distances.max()


def dH(A, B, T): # dH(T(A), B)
    return max(A.transform(T).asymm_dH(B),
               B.transform(T.invert()).asymm_dH(A))


def hausDist(A,B):
    m = sp.distance.directed_hausdorff(A, B)[0]
    n = sp.distance.directed_hausdorff(B, A)[0]
    return max(m,n)

In [3]:
%%time
n_dim = 3
n = 10000
n_A, n_B = n, n

# Generate points.
np.random.seed(10)
coords_A, coords_B = [np.random.rand(n, n_dim) for n in [n_A, n_B]]

# Construct the two point clouds.
A, B = (PointCloud(coords) for coords in [coords_A, coords_B])

CPU times: user 12.1 ms, sys: 22 µs, total: 12.1 ms
Wall time: 14.9 ms


#### Approximating Euclidean–Hausdorff distance between A and B:


In [19]:
%%time
additive_err_ub = .05 * max(map(lambda x: diam(x.coords), [A, B]))
print(f'desired additive error: {additive_err_ub:.5f}')

dEH, err_ub = approx_eucl_haus(coords_A, coords_B, proper_rigid=True, verbose=2)
print(f'dEH ∈ [{dEH - err_ub:.5f}, {dEH:.5f}]')

desired additive error: 0.08497
r=0.86303, max diam=1.69934
best_dH=inf, err_ub=inf, n_no_improv=0, Qs=[1]
updated depth range to 1-1
best_dH=0.31355, err_ub=0.31355, n_no_improv=0, Qs=[0, 64]
best_dH=0.21314, err_ub=0.21314, n_no_improv=0, Qs=[0, 63, 64]
best_dH=0.12944, err_ub=0.12944, n_no_improv=0, Qs=[0, 63, 63, 64]
best_dH=0.09628, err_ub=0.09628, n_no_improv=0, Qs=[0, 63, 63, 63, 64]
best_dH=0.08435, err_ub=0.08435, n_no_improv=0, Qs=[0, 63, 63, 63, 63, 64]
best_dH=0.07439, err_ub=0.07439, n_no_improv=0, Qs=[0, 63, 63, 63, 63, 63, 64]
dEH ∈ [0.00000, 0.07439]
CPU times: user 19.2 s, sys: 481 ms, total: 19.7 s
Wall time: 7.57 s


#### Testing Hausdorff distance between A and B:


In [25]:
%%time
T = Transformation([0, 0, 0], [0, 0, 0], False) # identity
dH(A, B, T)

CPU times: user 44.2 ms, sys: 0 ns, total: 44.2 ms
Wall time: 17.9 ms


0.08020952158971556

In [13]:
%%time
hausDist(A.coords, B.coords)

CPU times: user 87.9 ms, sys: 3.07 ms, total: 91 ms
Wall time: 89.4 ms


0.024009187809924486